Baseline Model 1: ridge_gridsearch

In [29]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.linear_model import Ridge

In [30]:
train_input = "new_data\mover_epic_final_train_features.csv"
test_input = "new_data\mover_epic_final_test_features.csv"

print("[Ridge GridSearch Pipeline] Loading data...")
if not os.path.exists(train_input) or not os.path.exists(test_input):
    raise FileNotFoundError("Missing essential upstream processed cohort files.")

train_df = pd.read_csv(train_input, encoding='utf-8')
test_df = pd.read_csv(test_input, encoding='utf-8')

if 'IN_OR_DTTM' in train_df.columns:
    train_df['IN_OR_DTTM'] = pd.to_datetime(train_df['IN_OR_DTTM'])
    train_df = train_df.sort_values('IN_OR_DTTM', ascending=True).reset_index(drop=True)

[Ridge GridSearch Pipeline] Loading data...


In [31]:
scaler_features = [
    'AGE', 'HEIGHT', 'WEIGHT',
    'SCHEDULED_START_HOUR',
    'SURGERY_DAY_OF_WEEK',
    'SURGERY_MONTH'
]

binary_features = [
    'SEX_CODE', 'ICU_ADMIN_FLAG_CODE', 'IS_MORNING_CASE_CODE',
    'Is_Hypertension', 'Is_Diabetes', 'Is_Cardiac'
]

cyclical_features = [
    'DOW_SIN', 'DOW_COS',
    'MONTH_SIN', 'MONTH_COS',
    'HOUR_SIN', 'HOUR_COS'
]

categorical_features = ['ASA_RATING_C']

procedure_cols = [c for c in train_df.columns if c.startswith("PRIMARY_PROCEDURE_NM_")]


In [32]:
print("[Ridge GridSearch] Engineering explicit domain-specific interaction metrics...")

# Run strict vectorization multiplication over training and testing partitions simultaneously
for df_block in [train_df, test_df]:
    # 1. Physiological Frailty Synergy Cross Term
    df_block['INTERACT_AGE_ASA'] = df_block['AGE'] * df_block['ASA_RATING_C']
    
    # 2. Chronological Operational Friction Cross Term
    df_block['INTERACT_HOUR_MORNING'] = df_block['SCHEDULED_START_HOUR'] * df_block['IS_MORNING_CASE_CODE']
    
    # 3. High-Acuity Downstream ICU Trajectory Cross Terms
    for proc_col in procedure_cols:
        new_interact_col = f"INTERACT_ICU_x_{proc_col}"
        df_block[new_interact_col] = df_block['ICU_ADMIN_FLAG_CODE'] * df_block[proc_col]

# Compile the newly generated interaction indicators list dynamically
interaction_cols = (
    ['INTERACT_AGE_ASA', 'INTERACT_HOUR_MORNING'] + 
    [f"INTERACT_ICU_x_{c}" for c in procedure_cols]
)

[Ridge GridSearch] Engineering explicit domain-specific interaction metrics...


In [33]:

feature_cols = sorted(
    scaler_features + cyclical_features + binary_features + categorical_features + procedure_cols + interaction_cols
)
leakage_exclusion = [
    'LOG_ID', 'MRN', 'CASE_ID', 'SEX', 'PRIMARY_PROCEDURE_NM', 'ASA_RATING', 'ICU_ADMIN_FLAG', 'IS_MORNING_CASE',
    'IN_OR_DTTM', 'OUT_OR_DTTM', 'AN_START_DATETIME', 'AN_STOP_DATETIME',
    'SURGERY_DATE', 'BIRTH_DATE', 'HOSP_ADMSN_TIME', 'HOSP_DISCH_TIME',
    'ACTUAL_DURATION', 'TARGET_LOG', 'DISCH_DISP'
]
for split_name, cohort_df in [('Training', train_df), ('Testing', test_df)]:
    active_cols = [col for col in cohort_df.columns if col not in leakage_exclusion]
    missing = set(feature_cols) - set(active_cols)
    extra = set(active_cols) - set(feature_cols)
    assert len(missing) == 0, f"Missing metrics in {split_name} schema contract: {missing}"
    assert len(extra) == 0, f"Unauthorized extra columns inside {split_name} block: {extra}"
    

In [34]:
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train_raw = train_df['ACTUAL_DURATION'].copy()
y_train_log = np.log1p(y_train_raw)
y_test_true = test_df['ACTUAL_DURATION'].copy()

print(f"   [Schema Alignment] Feature Space Uniform Size: {len(feature_cols)} continuous/discrete nodes.")
print(f"   [Skewness Check] Raw Target: {y_train_raw.skew():.4f} | Log Target Manifold: {y_train_log.skew():.4f}")

   [Schema Alignment] Feature Space Uniform Size: 3115 continuous/discrete nodes.
   [Skewness Check] Raw Target: 1.8508 | Log Target Manifold: 0.0061


In [35]:
print("[Ridge GridSearch] Constructing segmented ColumnTransformer pipeline")
continuous_features = scaler_features + cyclical_features
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=True, with_std=True))
])

bin_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0))
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer([
    ('num_flow', num_pipe, continuous_features),
    ('bin_flow', bin_pipe, binary_features),
    ('cat_flow', cat_pipe, categorical_features),
    ('proc_flow', 'passthrough', procedure_cols),
    ('interact_flow', 'passthrough', interaction_cols)
])

[Ridge GridSearch] Constructing segmented ColumnTransformer pipeline


In [36]:
pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', Ridge(random_state=42))
])

param_grid = {
    'model__alpha': np.logspace(-2, 3, 12)
}
tscv = TimeSeriesSplit(n_splits=5)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)


In [37]:
print("[Training] Running GridSearchCV hyperparameter searching over strict chronological splits")

grid.fit(X_train, y_train_log)

best_model = grid.best_estimator_

print(f"Best alpha: {grid.best_params_['model__alpha']}")

[Training] Running GridSearchCV hyperparameter searching over strict chronological splits
Best alpha: 0.6579332246575679


In [38]:
print("[Evaluation] Executing forward inference and inverse-scale prospective residuals metrics tracking...")

# Generate log-space forecasts for mathematical verification
pred_log_space = best_model.predict(X_test)
y_test_log_true = np.log1p(y_test_true)

# Reverse transform log forecasts back to authentic raw clinical minutes via monotonic exponential inversion
pred_raw_minutes = np.expm1(pred_log_space)

print("\n" + "="*75)
print("     FINAL PROSPECTIVE TEST RESULTS (2022 SEQUESTERED COHORT)")
print("="*75)
print("--- Absolute Clinical Minutes Scale (Downstream Tactical Reference) ---")
print(f"   Mean MAE (Mean Absolute Error)     : {mean_absolute_error(y_test_true, pred_raw_minutes):.2f} minutes")
print(f"   Mean RMSE (Root Mean Squared Error): {np.sqrt(mean_squared_error(y_test_true, pred_raw_minutes)):.2f} minutes")
print(f"   MedianAE (Median Absolute Error)   : {median_absolute_error(y_test_true, pred_raw_minutes):.2f} minutes")
print(f"   R2 Score (Variance Explained)      : {r2_score(y_test_true, pred_raw_minutes):.4f}")
print("-"*75)
print("--- Internal Log-Transformed Target Manifold (Structural Reference) ---")
print(f"   Log-MAE                            : {mean_absolute_error(y_test_log_true, pred_log_space):.4f}")
print(f"   Log-R2 Score                       : {r2_score(y_test_log_true, pred_log_space):.4f}")
print("="*75 + "\n")

[Evaluation] Executing forward inference and inverse-scale prospective residuals metrics tracking...

     FINAL PROSPECTIVE TEST RESULTS (2022 SEQUESTERED COHORT)
--- Absolute Clinical Minutes Scale (Downstream Tactical Reference) ---
   Mean MAE (Mean Absolute Error)     : 61.32 minutes
   Mean RMSE (Root Mean Squared Error): 97.51 minutes
   MedianAE (Median Absolute Error)   : 38.33 minutes
   R2 Score (Variance Explained)      : 0.5125
---------------------------------------------------------------------------
--- Internal Log-Transformed Target Manifold (Structural Reference) ---
   Log-MAE                            : 0.3063
   Log-R2 Score                       : 0.5883



Result before adding interaction features

[Training] Running GridSearchCV... Best alpha: 0.23101297000831592 

FINAL TEST RESULTS:

MAE  : 61.64 

RMSE : 98.13 

MedAE: 38.60 

R2   : 0.5063